# Maya: healthcare operations LLM — training & evaluation

Non-clinical hospital operations only. No diagnosis, treatment, prescription or triage.

Runtime: **Colab T4** (16GB). Steps below: clone repo, install pinned deps, validate the exact checkpoint, prepare the licensed dataset, run the baseline, smoke-test training, train, evaluate on the locked test set, produce artifacts, save to Drive.

Model: `empero-ai/Qwen3.8-4B-Distill` (Qwen3.5 architecture, Apache-2.0). Framework: Unsloth.

## 1. Clone the repository

In [ ]:
import os
REPO = os.environ.get("MAYA_REPO", "https://github.com/<YOUR_USER>/maya-healthcare-llm.git")
BRANCH = "main"
!rm -rf maya-healthcare-llm
!git clone -b {BRANCH} {REPO}
%cd maya-healthcare-llm

## 2. Install pinned dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps unsloth_zoo
# Qwen3.5 requires transformers v5; Unsloth uses v5 by default outside Colab.
!pip install -q --upgrade transformers
!pip install -q accelerate peft trl bitsandbytes flash-linear-attention causal-conv1d
!pip install -q -e .

## 3. GPU / RAM / software diagnostics

In [ ]:
import sys, torch, transformers
print("python", sys.version.split()[0])
print("torch", torch.__version__, "cuda", torch.version.cuda)
print("transformers", transformers.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("vram GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## 4. Validate the exact checkpoint + tokenizer

In [ ]:
from maya.model import inspect_checkpoint, DEFAULT_REPO
from maya.config import resolve_model_revision
print(inspect_checkpoint(DEFAULT_REPO))
print("resolved revision:", resolve_model_revision(DEFAULT_REPO))

## 5. Audit + prepare the licensed dataset

In [ ]:
from maya import data
manifest = data.prepare_dataset(seed=20241015, examples_per_category=520, out_dir="data/processed")
import json
print(json.dumps(manifest, indent=2))

## 6. Original-model baseline (E0)

In [ ]:
!python -m maya.cli evaluate --experiment-name E0 --max-examples 60

## 7. Training smoke test (proves load + fit before full run)

In [ ]:
import json
from maya import config, data, train
from maya.model import DEFAULT_REPO
cfgs = config.load_all_configs()
cfgs["training_t4"]["training"]["max_steps"] = 5  # smoke: just prove it loads and fits
tr = data.load_processed_split("data/processed", "train")[:64]
va = data.load_processed_split("data/processed", "validation")[:16]
res = train.run_training(DEFAULT_REPO, tr, va, "outputs/smoke", cfgs, resume=False)
print("smoke train OK:", res.final_checkpoint)

## 8. Configured training run (resumable)

In [ ]:
!python -m maya.cli train

## 9. Evaluate base vs adapter on the same locked test set (E0/E1)

In [ ]:
# E1: fine-tuned adapter. Point --adapter at the saved LoRA dir.
!python -m maya.cli evaluate --experiment-name E1 --adapter outputs/smoke

## 10. Reports, plots and metrics (per experiment id)

In [ ]:
# Reports/plots/metrics are written by the evaluate step to artifacts/<id>/.
!ls -R artifacts | head -50

## 11. Save adapters + reports to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copytree('artifacts', '/content/drive/MyDrive/maya/artifacts', dirs_exist_ok=True)
shutil.copytree('outputs', '/content/drive/MyDrive/maya/outputs', dirs_exist_ok=True)
print('saved to Drive')